[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LimitlessGreen/Audio-Workbench/blob/main/python-wrapper/demo_colab.ipynb)

# Audio Workbench Python Demo (Colab)

This notebook demonstrates how to use the `audio-workbench` Python wrapper to visualize audio as a spectrogram in Google Colab.


## 1. Install audio-workbench and dependencies

This cell installs the audio-workbench package and its dependencies. Colab may require a restart after installation.

In [ ]:
!pip install "audio-workbench[widget]"
# Optionally, install demo dependencies
# !pip install "audio-workbench[streamlit]"

## 2. Download a sample audio file

We download a Tree Pipit (*Anthus trivialis*) recording from [xeno-canto (XC1089065)](https://xeno-canto.org/1089065).

In [ ]:
from pathlib import Path
from urllib.request import urlopen, Request

XC_ID = 1089065
url = f"https://xeno-canto.org/{XC_ID}/download"

# Detect actual file extension from Content-Disposition / Content-Type
resp = urlopen(Request(url, headers={"User-Agent": "audio-workbench-demo"}))
content_disp = resp.headers.get("Content-Disposition", "")
if "filename=" in content_disp:
    fname = content_disp.split("filename=")[-1].strip(' "')
    ext = Path(fname).suffix or ".mp3"
else:
    ct = resp.headers.get("Content-Type", "")
    ext = {
        "audio/mpeg": ".mp3", "audio/wav": ".wav", "audio/x-wav": ".wav",
        "audio/ogg": ".ogg", "audio/flac": ".flac",
    }.get(ct.split(";")[0].strip(), ".mp3")

audio_path = Path(f"XC{XC_ID}{ext}")
if not audio_path.exists():
    audio_path.write_bytes(resp.read())
else:
    resp.close()

print(f"{audio_path}  ({audio_path.stat().st_size / 1024:.0f} KB)")

## 3a. Player (iframe mode — read-only, no Python sync)

`Player` wraps the iframe renderer and auto-displays in the notebook — no `HTML()` import needed.

In [ ]:
from audio_workbench import Player

Player(audio_path, viewMode="spectrogram", transportStyle="hero")

## 3b. Interactive Widget (bidirectional — annotations sync to Python)

`AudioWorkbenchWidget` uses `anywidget` to provide real-time sync between the JS player and Python. Draw annotations in the player — they appear in `w.annotations` and `w.spectrogram_labels` automatically.

In [ ]:
from audio_workbench import AudioWorkbenchWidget

w = AudioWorkbenchWidget(audio_path, viewMode="both")
w

### Read annotations back in Python

After drawing annotations in the player above, run the next cell to see them:

In [ ]:
# Waveform annotations (time regions)
print("Annotations:", w.annotations)

# Spectrogram labels (time × frequency boxes)
print("Spectrogram labels:", w.spectrogram_labels)

# Playback state
print(f"Time: {w.current_time:.2f}s / {w.duration:.2f}s, Playing: {w.playing}")

### Control playback & push annotations from Python

In [ ]:
# Play / pause / stop from Python
w.play()
# w.pause()
# w.stop()

# Push annotations programmatically
w.set_annotations([
    {"start": 1.0, "end": 2.5, "species": "Bird Call"},
    {"start": 4.0, "end": 5.0, "species": "Unknown"},
])

### Listen for events in real-time

In [ ]:
# Register a callback for new annotations
def on_new_annotation(event, detail):
    print(f"[{event}] {detail}")

w.on_event("annotationcreate", on_new_annotation)
w.on_event("spectrogramlabelcreate", on_new_annotation)
print("Callbacks registered. Draw an annotation in the player above!")

## 4. Widget Presets & Layouts

The `AudioWorkbenchWidget` accepts all `BirdNETPlayer` constructor options. Here are common configurations:

### Spectrogram-only (compact)

In [ ]:
# Spectrogram-only view, no sidebar, no overview — great for embedding
compact_spect = AudioWorkbenchWidget(
    audio_path,
    viewMode="spectrogram",
    showOverview=False,
    showStatusbar=False,
    showDisplayGain=False,
    iframe_height=350,
)
compact_spect

### Hero transport overlay

Minimal play button centered on the spectrogram — ideal for previews.

In [ ]:
# Hero overlay — big centered play button, spectrogram background
hero = AudioWorkbenchWidget(
    audio_path,
    viewMode="spectrogram",
    transportStyle="hero",
    transportOverlay=True,
    showFileOpen=False,
    showZoom=False,
    showFFTControls=False,
    showDisplayGain=False,
    showViewToggles=False,
    showStatusbar=False,
    showOverview=False,
    iframe_height=300,
)
hero

### Waveform-only

In [ ]:
# Waveform-only view — lightweight, no spectrogram computation
waveform = AudioWorkbenchWidget(
    audio_path,
    viewMode="waveform",
    showDisplayGain=False,
    showFFTControls=False,
    iframe_height=250,
)
waveform

### Custom label taxonomy

Define species presets with colors and keyboard shortcuts for fast annotation.

In [ ]:
# Custom taxonomy — draw a label, then press 1/2/3 to classify
taxonomy_widget = AudioWorkbenchWidget(
    audio_path,
    viewMode="both",
    labelTaxonomy=[
        {"name": "Anthus trivialis",  "color": "#22c55e", "shortcut": "1"},
        {"name": "Turdus merula",     "color": "#3b82f6", "shortcut": "2"},
        {"name": "Unknown",           "color": "#f59e0b", "shortcut": "3"},
    ],
)
taxonomy_widget

### Pre-loaded annotations

Inject existing annotation data at creation time — useful for reviewing model predictions.

In [ ]:
# Create widget and immediately push annotations (e.g. BirdNET detections)
review = AudioWorkbenchWidget(audio_path, viewMode="both")
review.set_annotations([
    {"start": 3.0, "end": 6.0, "species": "Anthus trivialis", "color": "#22c55e"},
    {"start": 15.0, "end": 19.0, "species": "Anthus trivialis", "color": "#22c55e"},
    {"start": 40.0, "end": 44.0, "species": "Anthus trivialis", "color": "#22c55e"},
])
review

## 5. (Optional) Try your own audio

You can upload your own audio file and visualize it by replacing the file path in the code above.

---

**This notebook is ready to run in Google Colab.**

- [Project repository](https://github.com/LimitlessGreen/Audio-Workbench)
- [PyPI package](https://pypi.org/project/audio-workbench/)